## RUN ALL MODELS EXCEPT OVIS

In [ ]:
# run_notebook_grid.py
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError
import jupyter_client
from pathlib import Path
import copy
from itertools import product

# === CONFIG ===
INPUT_NOTEBOOK = Path("general_notebook.ipynb")
N_SIMULATIONS  = 32
TASKS = ["asch_lines", "color_recognition", "dots_estimation"]

MODELS = {
    "qwen":   ["qwen_3B", "qwen_7B", "qwen_32B", "qwen2_7B"],
    "gemma":  ["gemma_4B", "gemma_12B", "gemma_27B"],
    "mistral":["mistral_24B"],
}

def make_parameters_cell(family, model_label, task, n_simulations):
    code = (
        f"family = {repr(family)}\n"
        f"model_label = {repr(model_label)}\n"
        f"task = {repr(task)}\n"
        f"n_simulations = {int(n_simulations)}\n"
    )
    cell = nbformat.v4.new_code_cell(code)
    cell.metadata.setdefault("tags", []).append("parameters")
    return cell

def inject_or_replace_parameters(nb, cell):
    """Sostituisce una cella con tag 'parameters' se esiste; altrimenti inserisce in testa."""
    for i, c in enumerate(nb.cells):
        if "tags" in c.metadata and "parameters" in c.metadata["tags"]:
            nb.cells[i] = cell
            return
    nb.cells.insert(0, cell)

def run_once(nb_src, family, model_label, task, n_simulations):
    # copia del notebook sorgente
    nb = copy.deepcopy(nb_src)

    # inietta/aggiorna cella parametri
    params_cell = make_parameters_cell(family, model_label, task, n_simulations)
    inject_or_replace_parameters(nb, params_cell)

    print(f"👉 Run: family={family:<7} | model={model_label:<12} | task={task}")

    # avvia kernel nuovo
    km = jupyter_client.KernelManager()
    km.start_kernel()
    kc = km.client()
    kc.start_channels()
    kc.wait_for_ready()

    try:
        client = NotebookClient(
            nb,
            km=km,
            kernel_name="python3",
            timeout=None,
            resources={"metadata": {"path": str(INPUT_NOTEBOOK.parent)}},
        )
        client.execute()  # esegue tutte le celle in memoria
        print(f"✅ Completato: {family}/{model_label}/{task}")
    except CellExecutionError as e:
        print(f"❌ Errore con {family}/{model_label}/{task}:\n{e}")
    finally:
        kc.stop_channels()
        km.shutdown_kernel(now=True)

def main():
    # carica notebook sorgente UNA volta
    nb_src = nbformat.read(INPUT_NOTEBOOK, as_version=4)

    # tutte le combinazioni richieste
    combos = []
    for family, models in MODELS.items():
        for model_label, task in product(models, TASKS):
            combos.append((family, model_label, task))

    print(f"Totale run: {len(combos)}")
    for family, model_label, task in combos:
        run_once(nb_src, family, model_label, task, N_SIMULATIONS)

if __name__ == "__main__":
    main()

## RUN SIMULATIONS FOR OVIS MODELS

In [ ]:
# === CONFIG ===
INPUT_NOTEBOOK = Path("general_notebook_ovis.ipynb")
TASKS = ["asch_lines", "color_recognition", "dots_estimation"]

MODELS = {
    "ovis":   ["ovis_4B", "ovis_8B", "ovis_16B", "ovis_34B"],
}

def make_parameters_cell(family, model_label, task, n_simulations):
    code = (
        f"family = {repr(family)}\n"
        f"model_label = {repr(model_label)}\n"
        f"task = {repr(task)}\n"
        f"n_simulations = {int(n_simulations)}\n"
    )
    cell = nbformat.v4.new_code_cell(code)
    cell.metadata.setdefault("tags", []).append("parameters")
    return cell

def inject_or_replace_parameters(nb, cell):
    """Sostituisce una cella con tag 'parameters' se esiste; altrimenti inserisce in testa."""
    for i, c in enumerate(nb.cells):
        if "tags" in c.metadata and "parameters" in c.metadata["tags"]:
            nb.cells[i] = cell
            return
    nb.cells.insert(0, cell)

def run_once(nb_src, family, model_label, task, n_simulations):
    # copia del notebook sorgente
    nb = copy.deepcopy(nb_src)

    # inietta/aggiorna cella parametri
    params_cell = make_parameters_cell(family, model_label, task, n_simulations)
    inject_or_replace_parameters(nb, params_cell)

    print(f"👉 Run: family={family:<7} | model={model_label:<12} | task={task}")

    # avvia kernel nuovo
    km = jupyter_client.KernelManager()
    km.start_kernel()
    kc = km.client()
    kc.start_channels()
    kc.wait_for_ready()

    try:
        client = NotebookClient(
            nb,
            km=km,
            kernel_name="python3",
            timeout=None,
            resources={"metadata": {"path": str(INPUT_NOTEBOOK.parent)}},
        )
        client.execute()  # esegue tutte le celle in memoria
        print(f"✅ Completato: {family}/{model_label}/{task}")
    except CellExecutionError as e:
        print(f"❌ Errore con {family}/{model_label}/{task}:\n{e}")
    finally:
        kc.stop_channels()
        km.shutdown_kernel(now=True)

def main():
    # carica notebook sorgente UNA volta
    nb_src = nbformat.read(INPUT_NOTEBOOK, as_version=4)

    # tutte le combinazioni richieste
    combos = []
    for family, models in MODELS.items():
        for model_label, task in product(models, TASKS):
            combos.append((family, model_label, task))

    print(f"Totale run: {len(combos)}")
    for family, model_label, task in combos:
        run_once(nb_src, family, model_label, task, N_SIMULATIONS)

if __name__ == "__main__":
    main()